# MaldiAMRKit - Custom Components

MaldiAMRKit dispatches several of its building blocks by name. Three of those name tables are open for extension, so you can plug in your own component and use it exactly as if it shipped with the package:

| Registry | Register with | Used by |
| --- | --- | --- |
| Spectral metrics | `register_spectral_metric` | `spectral_distance`, `pairwise_distances`, `DriftMonitor` |
| Preprocessing transformers | `register_transformer` | `PreprocessingPipeline` serialisation |
| Binning methods | `register_binning_method` | `bin_spectrum`, `MaldiSpectrum.bin` |

This notebook registers a custom metric and a custom transformer end-to-end, and shows how the built-ins are protected.

## Import Libraries

In [1]:
import warnings

import numpy as np
import pandas as pd

from maldiamrkit import MaldiSet
from maldiamrkit.preprocessing import (
    ClipNegatives,
    PreprocessingPipeline,
    list_binning_methods,
    list_transformers,
    register_transformer,
    unregister_transformer,
)
from maldiamrkit.similarity import (
    extract_mz_intensity,
    list_spectral_metrics,
    pairwise_distances,
    register_spectral_metric,
    spectral_distance,
    unregister_spectral_metric,
)

/home/ettore/Documents/MaldiAMRKit/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
data = MaldiSet.from_directory(
    "../data/",
    "../data/metadata/metadata.csv",
    aggregate_by=dict(antibiotics="Drug"),
)
X = data.X
print(f"Binned spectra shape: {X.shape}")

Binned spectra shape: (29, 6000)


## Part 1 - A custom spectral metric

### Writing the function

A metric is any callable `fn(spec_a, spec_b) -> float`. It receives the two spectra **exactly as the caller passed them**: a `MaldiSpectrum`, a `(mass, intensity)` DataFrame, or a bare 1-D intensity vector. Rather than assuming one of those shapes, normalise both arguments with `extract_mz_intensity`, which is the same helper the built-in metrics use. It returns `(mz, intensity)`, with `mz` being `None` for binned vectors.

Below: the Manhattan (L1) distance between binned intensity vectors. Defining it at module level (or in a notebook cell, as here) keeps it serialisable, which matters for the parallel section further down.

In [3]:
def manhattan(spec_a, spec_b):
    """L1 distance between two binned intensity vectors."""
    _, a = extract_mz_intensity(spec_a)
    _, b = extract_mz_intensity(spec_b)
    return float(np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float)).sum())


# Sanity check before registering anything
manhattan(np.array([1.0, 2.0, 3.0]), np.array([1.0, 5.0, 3.0]))

3.0

### Registering it

In [4]:
register_spectral_metric("manhattan", manhattan)
list_spectral_metrics()

['cosine',
 'dtw',
 'manhattan',
 'pearson',
 'spectral_contrast_angle',
 'wasserstein']

The name is now accepted anywhere a metric name is, starting with `spectral_distance`:

In [5]:
d_manhattan = spectral_distance(X.iloc[0], X.iloc[1], metric="manhattan")
d_cosine = spectral_distance(X.iloc[0], X.iloc[1], metric="cosine")
print(f"manhattan: {d_manhattan:.6f}")
print(f"cosine:    {d_cosine:.6f}")

manhattan: 0.572700
cosine:    0.141512


### Using it in parallel

`pairwise_distances` resolves the metric function in the calling process and hands the function itself to its workers, so a custom metric works at any `n_jobs`. (Worker processes get a fresh import of the package and therefore do **not** inherit your registration, which is why the name alone would not be enough.)

In [6]:
D = pairwise_distances(X, metric="manhattan", n_jobs=-1)

print(f"shape:      {D.shape}")
print(f"symmetric:  {np.allclose(D, D.T)}")
print(f"zero diag:  {np.allclose(np.diag(D), 0.0)}")
pd.DataFrame(D[:4, :4], index=X.index[:4], columns=X.index[:4]).round(4)

shape:      (29, 29)
symmetric:  True
zero diag:  True


,10s,11s,12s,13s
10s,0.0000,0.5727,0.5384,0.5658
11s,0.5727,0.0000,0.4451,0.5807
12s,0.5384,0.4451,0.0000,0.5125
13s,0.5658,0.5807,0.5125,0.0000


Because the metric is resolved by name, anything that forwards a metric name inherits it too - `DriftMonitor(metric="manhattan")`, for instance, needs no extra wiring.

### The built-ins are protected

Replacing a shipped metric is possible but never accidental, and it is reversible: unregistering an overridden built-in restores the default implementation. The built-in names themselves can never be removed.

In [7]:
try:
    register_spectral_metric("cosine", manhattan)
except ValueError as exc:
    print("register:", exc)

try:
    unregister_spectral_metric("cosine")
except ValueError as exc:
    print("unregister:", exc)

# With an explicit override the swap succeeds, and unregistering the
# overridden name restores the default implementation:
register_spectral_metric("cosine", manhattan, override=True)
print("overridden:", spectral_distance(X.iloc[0], X.iloc[1], metric="cosine"))
unregister_spectral_metric("cosine")
print("restored:  ", spectral_distance(X.iloc[0], X.iloc[1], metric="cosine"))

register: 'cosine' is a built-in spectral metric. Pass override=True to replace it.
unregister: Cannot unregister built-in spectral metric 'cosine'. Built-in metrics are ['cosine', 'dtw', 'pearson', 'spectral_contrast_angle', 'wasserstein'].
overridden: 0.5726999305940348
restored:   0.1415117096696954


Registrations are process-global, so clean up anything you no longer need:

In [8]:
unregister_spectral_metric("manhattan")
list_spectral_metrics()

['cosine', 'dtw', 'pearson', 'spectral_contrast_angle', 'wasserstein']

## Part 2 - A custom transformer

### Why register at all

A `PreprocessingPipeline` already accepts *any* object that is callable on a spectrum DataFrame - no registration needed to use one in memory. Registration closes the **serialisation** gap: it is what lets `from_dict` / `from_json` / `from_yaml` rebuild your transformer from a saved config.

Two rules make a transformer round-trip:

1. `to_dict()` returns `{"name": <registered name>, **constructor_kwargs}`.
2. Every key it returns other than `"name"` is a valid argument to `__init__`, because reconstruction is `cls(**kwargs)`.

The example is a power transform, generalising the built-in `SqrtTransform` to an arbitrary exponent for variance stabilisation.

In [9]:
class PowerTransform:
    """Variance-stabilising power transform: intensity -> intensity ** exponent."""

    def __init__(self, exponent: float = 0.5):
        self.exponent = exponent

    def __call__(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply the power transform to the spectrum."""
        df = df.copy()
        df["intensity"] = np.power(df["intensity"].clip(lower=0.0), self.exponent)
        return df

    def to_dict(self) -> dict:
        """Serialize transformer to a dictionary."""
        return {"name": "PowerTransform", "exponent": self.exponent}

    def __repr__(self) -> str:
        return f"PowerTransform(exponent={self.exponent})"

### Serialising before registering

Build a pipeline with it and serialise. The config is still produced, but MaldiAMRKit warns that it could not be read back - the failure is surfaced at save time rather than on whatever machine tries to load it later.

In [10]:
pipe = PreprocessingPipeline(
    [
        ("clip", ClipNegatives()),
        ("power", PowerTransform(0.4)),
    ]
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    config = pipe.to_dict()

for w in caught:
    print(f"{w.category.__name__}: {w.message}")

Loading that config fails, and the error names the fix:

In [11]:
try:
    PreprocessingPipeline.from_dict(config)
except ValueError as exc:
    print(f"ValueError: {exc}")

ValueError: 'PowerTransform' is not a registered transformer. Use one of ['ClipNegatives', 'ConvexHullBaseline', 'LogTransform', 'MedianBaseline', 'MedianNormalizer', 'MovingAverageSmooth', 'MzMultiTrimmer', 'MzTrimmer', 'PQNNormalizer', 'SNIPBaseline', 'SavitzkyGolaySmooth', 'SqrtTransform', 'TICNormalizer', 'TopHatBaseline'] or register a custom transformer with register_transformer().


### Registering it

In [12]:
register_transformer("PowerTransform", PowerTransform)
[name for name in list_transformers() if "Transform" in name]

['LogTransform', 'PowerTransform', 'SqrtTransform']

Now the same pipeline serialises cleanly and rebuilds exactly:

In [13]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    config = pipe.to_dict()

print(f"warnings: {len(caught)}")
config

warnings: 0


{'steps': [{'step_name': 'clip', 'name': 'ClipNegatives'},
  {'step_name': 'power', 'name': 'PowerTransform', 'exponent': 0.4}]}

In [14]:
rebuilt = PreprocessingPipeline.from_dict(config)
print(rebuilt)
print(f"\nexponent preserved: {rebuilt.get_step('power').exponent}")

PreprocessingPipeline([
  ('clip', ClipNegatives()),
  ('power', PowerTransform(exponent=0.4))
])

exponent preserved: 0.4


The rebuilt pipeline is behaviourally identical to the original:

In [15]:
raw = data.spectra[0].raw
np.allclose(pipe(raw)["intensity"], rebuilt(raw)["intensity"])

True

### Round-tripping through a file

In [16]:
pipe.to_json("custom_pipeline.json")
from_file = PreprocessingPipeline.from_json("custom_pipeline.json")
print(from_file)

PreprocessingPipeline([
  ('clip', ClipNegatives()),
  ('power', PowerTransform(exponent=0.4))
])


This is the practical payoff: a preprocessing configuration containing your own transformer can now be version-controlled, shipped alongside a dataset, or handed to `DatasetBuilder` via a `ProcessingHandler`, and any environment that registers `PowerTransform` will reproduce it.

### Cleaning up

In [17]:
from pathlib import Path

Path("custom_pipeline.json").unlink()
unregister_transformer("PowerTransform")
print(f"{len(list_transformers())} transformers registered (built-ins only)")

14 transformers registered (built-ins only)


## Part 3 - Discovery and protection

Each registry exposes a `list_*` function returning built-ins plus whatever the current process has registered. Useful when a config fails to load and you need to see what the interpreter actually knows about.

In [18]:
print("spectral metrics :", list_spectral_metrics())
print("binning methods  :", list_binning_methods())
print("transformers     :", len(list_transformers()), "registered")

spectral metrics : ['cosine', 'dtw', 'pearson', 'spectral_contrast_angle', 'wasserstein']
binning methods  : ['adaptive', 'custom', 'proportional', 'uniform']
transformers     : 14 registered


The protection rules are the same across the metric and transformer registries:

- **Registering a new name** - always allowed.
- **Re-registering your own name** - allowed silently, so iterating in a notebook is painless.
- **Registering over a built-in** - refused unless you pass `override=True`.
- **Unregistering a built-in** - restores the default if the name was overridden, and is refused otherwise, so a built-in name never disappears.